In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_df=pd.read_csv('/kaggle/input/traindata/train_Kaggle_Comp.csv')
test_df=pd.read_csv('/kaggle/input/testdata/test_Kaggle_Comp.csv')
sample_df=pd.read_csv('/kaggle/input/sample/sample_submission_Kaggle_Comp.csv')

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
sample_df.shape

In [ ]:
train_df.columns

In [ ]:
test_df.columns

In [ ]:
sample_df.columns

In [ ]:
train_df.select_dtypes(include=['int64','float64']).columns

In [ ]:
train_df.select_dtypes(include=['object']).columns

In [ ]:
train_df.isnull().sum()

In [ ]:
test_df.isnull().sum()

In [ ]:
# Basic statistics of target variable
train_df['burnout_score'].describe()


In [ ]:
# Separate input features (X) and target (y)

X = train_df.drop(columns=['burnout_score', 'employee_id'])
y = train_df['burnout_score']


In [ ]:
X.columns

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print(X_train.shape, X_val.shape)



In [ ]:

print(y_train.shape, y_val.shape)


In [ ]:
X_train.select_dtypes(include = ['object','bool','category']).columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_cols = [
    'gender', 'education_level', 'job_role', 'company_type',
    'deadline_frequency', 'remote_work', 'physical_activity', 'health_issues'
]

numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)


In [ ]:
print(X_train_encoded.shape)



In [ ]:
print(X_val_encoded.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Categorical and numerical columns
categorical_cols = [
    'gender', 'education_level', 'job_role', 'company_type',
    'deadline_frequency', 'remote_work', 'physical_activity', 'health_issues'
]

numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

# Create preprocessor with scaling
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Fit on training data and transform train & validation
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)


In [ ]:
print(X_train_scaled.shape)



In [ ]:
print(X_val_scaled.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Initialize model
lin_reg = LinearRegression()

# Train model
lin_reg.fit(X_train_scaled, y_train)

# Predict on validation set
y_val_pred = lin_reg.predict(X_val_scaled)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print("Validation RMSE:", rmse)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Initialize Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train model
rf.fit(X_train_scaled, y_train)

# Predict on validation set
y_val_pred_rf = rf.predict(X_val_scaled)

# Calculate RMSE
rf_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))

print("Random Forest Validation RMSE:", rf_rmse)


In [ ]:
from sklearn.linear_model import LinearRegression

# Use full training data
X_full = X
y_full = y

# Refit preprocessor on full data
X_full_scaled = preprocessor.fit_transform(X_full)

# Train final Linear Regression model
final_model = LinearRegression()
final_model.fit(X_full_scaled, y_full)


In [ ]:
# Prepare test features (remove ID)
X_test = test_df.drop(columns=['employee_id'])

# Apply same preprocessing (DO NOT fit again)
X_test_scaled = preprocessor.transform(X_test)

# Predict burnout scores
test_predictions = final_model.predict(X_test_scaled)


In [ ]:
test_predictions[:5]


In [ ]:
import numpy as np

# Optional safety
test_predictions = np.clip(test_predictions, 0, 1)

submission = sample_df.copy()
submission['burnout_score'] = test_predictions

submission.to_csv('submission.csv', index=False)


In [ ]:
submission


In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # Fit preprocessing on fold-train data
    X_tr_scaled = preprocessor.fit_transform(X_tr)
    X_va_scaled = preprocessor.transform(X_va)
    
    # Train Linear Regression
    model = LinearRegression()
    model.fit(X_tr_scaled, y_tr)
    
    # Predict & RMSE
    y_va_pred = model.predict(X_va_scaled)
    rmse = np.sqrt(mean_squared_error(y_va, y_va_pred))
    rmse_scores.append(rmse)
    
    print(f"Fold {fold} RMSE: {rmse:.5f}")

print("\nAverage CV RMSE:", np.mean(rmse_scores))
print("Std Dev RMSE:", np.std(rmse_scores))


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    
    # Fit preprocessing on fold-train
    X_tr_scaled = preprocessor.fit_transform(X_tr)
    
    # Prepare test data
    X_test = test_df.drop(columns=['employee_id'])
    X_test_scaled = preprocessor.transform(X_test)
    
    # Train model
    model = LinearRegression()
    model.fit(X_tr_scaled, y_tr)
    
    # Predict on test
    fold_test_preds = model.predict(X_test_scaled)
    test_preds += fold_test_preds
    
    print(f"Fold {fold} test prediction done")

# Average predictions
test_preds /= 5


In [ ]:
submission_cv = sample_df.copy()
submission_cv['burnout_score'] = np.clip(test_preds, 0, 1)

submission_cv.to_csv('submission_cv.csv', index=False)


In [ ]:
submission_cv.to_csv('submission.csv', index=False)


In [ ]:
from sklearn.linear_model import Ridge
import numpy as np

# Train Ridge on FULL training data
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_full_scaled, y_full)


In [ ]:
# Predict using final Linear Regression model
linear_test_preds = final_model.predict(X_test_scaled)

# Predict using Ridge Regression model
ridge_test_preds = ridge_final.predict(X_test_scaled)

# Safety clipping (keep values in [0, 1])
linear_test_preds = np.clip(linear_test_preds, 0, 1)
ridge_test_preds = np.clip(ridge_test_preds, 0, 1)

# Quick sanity check
print("Linear preds sample:", linear_test_preds[:5])
print("Ridge preds sample:", ridge_test_preds[:5])


In [ ]:
# Define safe weight combinations
weights = [
    (0.90, 0.10),
    (0.85, 0.15),
    (0.80, 0.20)
]

ensemble_predictions = {}

for w_lin, w_ridge in weights:
    blended_preds = (w_lin * linear_test_preds) + (w_ridge * ridge_test_preds)
    blended_preds = np.clip(blended_preds, 0, 1)
    
    key = f"lin_{int(w_lin*100)}_ridge_{int(w_ridge*100)}"
    ensemble_predictions[key] = blended_preds
    
    print(f"Created ensemble: {key}")


In [ ]:
# Select safest ensemble
final_ensemble_preds = ensemble_predictions['lin_90_ridge_10']

# Create submission
submission_ensemble = sample_df.copy()
submission_ensemble['burnout_score'] = final_ensemble_preds

submission_ensemble.to_csv('submission.csv', index=False)


In [ ]:
# Round predictions to 4 decimals
rounded_preds = np.round(linear_test_preds, 4)

# Create submission
submission_round = sample_df.copy()
submission_round['burnout_score'] = np.clip(rounded_preds, 0, 1)

submission_round.to_csv('submission.csv', index=False)


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np


In [ ]:
X_full_scaled = preprocessor.fit_transform(X)


In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_full_scaled, y)


In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_full_scaled, y)


In [ ]:
gbr_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gbr_model.fit(X_full_scaled, y)


In [ ]:
X_test = test_df.drop(columns=["employee_id"])
X_test_scaled = preprocessor.transform(X_test)


In [ ]:
lin_preds = lin_model.predict(X_test_scaled)
ridge_preds = ridge_model.predict(X_test_scaled)
gbr_preds = gbr_model.predict(X_test_scaled)


In [ ]:
lin_preds = np.clip(lin_preds, 0, 1)
ridge_preds = np.clip(ridge_preds, 0, 1)
gbr_preds = np.clip(gbr_preds, 0, 1)


In [ ]:
ensemble_preds = (
    0.7 * lin_preds +
    0.2 * ridge_preds +
    0.1 * gbr_preds
)




In [ ]:
ensemble_preds = np.clip(ensemble_preds, 0, 1)


In [ ]:
submission = sample_df.copy()
submission["burnout_score"] = ensemble_preds
submission.to_csv("submission.csv", index=False)
